In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
import glob

In [ ]:
# Файлы evaluation_*.csv
eval_files = glob.glob('evaluation_*.csv')
print("Найдены файлы:", eval_files)

# Метрики для анализа
metrics = ['smape', 'mase', 'rmsse']

Найдены файлы: ['evaluation_Lumpy.csv', 'evaluation_Erratic.csv', 'evaluation_all.csv', 'evaluation_Intermittent.csv', 'evaluation_Smooth.csv']


In [ ]:
# Конфигурация: baseline и список кандидатов с флагом better (True = кандидат лучше baseline)
comparison_config = {
    'all': {
        'baseline': 'AutoARIMA',
        'candidates': [
            {'model': 'AutoNHITS', 'better': False},
            {'model': 'AutoPatchTST', 'better': False},
            {'model': 'NBEATS', 'better': False},
            {'model': 'AutoTimesNet', 'better': False},
            {'model': 'SES', 'better': False},
            {'model': 'SESOpt', 'better': False},
            {'model': 'AutoNHITS', 'better': True},
            {'model': 'AutoPatchTST', 'better': True}
        ]
    },
    'Smooth': {
        'baseline': 'AutoARIMA',
        'candidates': [
            {'model': 'NBEATS', 'better': True},
            {'model': 'AutoPatchTST', 'better': True},
            {'model': 'NHITS', 'better': True},
            {'model': 'TFT', 'better': True},
            {'model': 'DeepAR', 'better': True},
            {'model': 'SES', 'better': True}
        ]
    },
    'Erratic': {
        'baseline': 'AutoARIMA',
        'candidates': [
            {'model': 'AutoPatchTST', 'better': True},
            {'model': 'NBEATS', 'better': True},
            {'model': 'NHITS', 'better': True},
            {'model': 'AutoTimesNet', 'better': True},
            {'model': 'SES', 'better': True},
            {'model': 'SESOpt', 'better': True}
        ]
    },
    'Intermittent': {
        'baseline': 'AutoARIMA',
        'candidates': [
            {'model': 'Chronos_tiny', 'better': False},
            {'model': 'SES', 'better': False},
            {'model': 'SESOpt', 'better': False},
            {'model': 'AutoPatchTST', 'better': False},
            {'model': 'NHITS', 'better': False},
            {'model': 'Chronos_tiny', 'better': True},
            {'model': 'SES', 'better': True},
            {'model': 'SESOpt', 'better': True},
        ]
    },
    'Lumpy': {
        'baseline': 'AutoARIMA',
        'candidates': [
            {'model': 'Chronos_tiny', 'better': False},
            {'model': 'AutoPatchTST', 'better': False},
            {'model': 'SESOpt', 'better': False},
            {'model': 'DeepAR', 'better': False},
            {'model': 'TFT', 'better': False},
             {'model': 'Chronos_tiny', 'better': True},
            {'model': 'AutoPatchTST', 'better': True},
        ]
    }
}

In [ ]:
all_results = []

for file in eval_files:
    class_name = file.replace('evaluation_', '').replace('.csv', '')
    if class_name not in comparison_config:
        continue
    config = comparison_config[class_name]
    baseline = config['baseline']
    candidates = config['candidates']
    df = pd.read_csv(file)

    if baseline not in df.columns:
        print(f"В {file} отсутствует baseline {baseline}")
        continue

    for candidate_info in candidates:
        candidate = candidate_info['model']
        better = candidate_info['better']
        if candidate not in df.columns:
            print(f"В {file} отсутствует модель {candidate}")
            continue

        for metric in metrics:
            sub = df[df['metric'] == metric]
            if sub.empty:
                print(f"В {file} нет метрики {metric}")
                continue

            vals_baseline = sub[baseline].values
            vals_candidate = sub[candidate].values
            mask = ~(np.isnan(vals_baseline) | np.isnan(vals_candidate))
            if np.sum(mask) < 2:
                print(f"Недостаточно пар для {class_name}, {metric}, {candidate}")
                continue

            vals_baseline_clean = vals_baseline[mask]
            vals_candidate_clean = vals_candidate[mask]

            # Выбор направления теста
            try:
                if better:
                    # проверяем, что ошибки кандидата меньше ошибок baseline
                    stat, p = wilcoxon(vals_candidate_clean, vals_baseline_clean, alternative='less')
                    model_better = candidate
                    model_worse = baseline
                else:
                    # проверяем, что ошибки кандидата больше ошибок baseline
                    stat, p = wilcoxon(vals_candidate_clean, vals_baseline_clean, alternative='greater')
                    model_better = baseline
                    model_worse = candidate
            except Exception as e:
                print(f"Ошибка для {class_name}, {metric}, {candidate}: {e}")
                continue

            all_results.append({
                'class': class_name,
                'metric': metric,
                'model_better': model_better,
                'model_worse': model_worse,
                'n_series': len(vals_baseline_clean),
                'p_value': p,
            })



/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:72: RuntimeWarning: invalid value encountered in subtract
  d = x - y
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:72: RuntimeWarning: invalid value encountered in subtract
  d = x - y
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:72: RuntimeWarning: invalid value encountered in subtract
  d = x - y
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:72: RuntimeWarning: invalid value encountered in subtract
  d = x - y
/usr/local/lib/python3.12/dist-packages/scipy/stats/_wilcoxon.py:72: RuntimeWarning: invalid value encountered in subtract
  d = x - y


In [ ]:
# Коррекция p-значений (Бонферрони)
if all_results:
    df_res = pd.DataFrame(all_results)
    _, p_corr, _, _ = multipletests(df_res['p_value'].values, alpha=0.05, method='bonferroni')
    df_res['p_corrected'] = p_corr
    df_res['significant_corrected'] = df_res['p_corrected'] < 0.05

    print("\n=== РЕЗУЛЬТАТЫ КРИТЕРИЯ ВИЛКОКСОНА ===\n")
    print(df_res[['class', 'metric', 'model_better', 'model_worse', 'n_series', 'p_value', 'p_corrected', 'significant_corrected']].to_string(index=False))

    df_res.to_csv('wilcoxon_all_pairs.csv', index=False)
    print("\nРезультаты сохранены в 'wilcoxon_all_pairs.csv'")
else:
    print("Нет данных для сравнения.")


=== РЕЗУЛЬТАТЫ КРИТЕРИЯ ВИЛКОКСОНА (ВСЕ МОДЕЛИ) ===

       class metric model_better  model_worse  n_series      p_value  p_corrected  significant_corrected
       Lumpy  smape    AutoARIMA Chronos_tiny       145 1.586573e-20 1.665901e-18                   True
       Lumpy   mase    AutoARIMA Chronos_tiny       145 4.404749e-01 1.000000e+00                  False
       Lumpy  rmsse    AutoARIMA Chronos_tiny       145 1.790673e-01 1.000000e+00                  False
       Lumpy  smape    AutoARIMA AutoPatchTST       145 8.888246e-16 9.332658e-14                   True
       Lumpy   mase    AutoARIMA AutoPatchTST       145 9.489038e-01 1.000000e+00                  False
       Lumpy  rmsse    AutoARIMA AutoPatchTST       145 9.134431e-01 1.000000e+00                  False
       Lumpy  smape    AutoARIMA       SESOpt       145 1.394678e-14 1.464412e-12                   True
       Lumpy   mase    AutoARIMA       SESOpt       145 9.156780e-03 9.614619e-01                  False
 